# ORIE5750 Final

**Team Members:** Yui To Wong, Yujie Gao, Steven Yang

We were unable to upload the full zip to Gradescope without surpassing size limits. To replicate, place each .parquet file in a folder data/. The data/ folder should be in the same directory as the notebook.


## Setup

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score
from arch import arch_model
import xgboost as xgb
import warnings

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

warnings.filterwarnings('ignore')


## Feature Engineering

In [2]:
class VolatilityFeatures:
    def __init__(self, anchor_tz: str = "America/New_York"):
        self.anchor_tz = anchor_tz

    def build_daily_table(self, bars: pd.DataFrame, freq: str = "5min") -> pd.DataFrame:
        if len(bars) == 0: return pd.DataFrame()
        df = bars.copy()
        
        # 1. Timezone & Sorting - from UTC to ET for clarity and market conventions
        if 'ts' in df.columns:
            df["ts"] = pd.to_datetime(df["ts"], utc=True)
        elif isinstance(df.index, pd.DatetimeIndex):
            df = df.reset_index().rename(columns={df.index.name: 'ts'})
            df["ts"] = pd.to_datetime(df["ts"], utc=True)
            
        df = df.sort_values("ts")
        
        # 2. Maintenance Filter (17:00 - 17:59 ET)
        ts_local = df["ts"].dt.tz_convert(self.anchor_tz)
        
        # Identify maintenance bars (Hour == 17) and drop any bars to ensure clean data
        mask_maintenance = ts_local.dt.hour == 17
        n_dropped = mask_maintenance.sum()
        print(f"Maintenance Filter: Dropped {n_dropped} bars found between 17:00-17:59 ET.")
        
        # Filter dataset
        df_clean = df[~mask_maintenance].copy()
        ts_clean = df_clean["ts"].dt.tz_convert(self.anchor_tz)
        
        # 3. Session Grouping for 1-minute OHLCV bars (18:00 open T-1 to 16:59 close T)
        # Logic: If hour >= 18, it belongs to the NEXT calendar day.
        df_clean["day"] = (ts_clean + pd.Timedelta(hours=6)).dt.floor("D")
        
        # Intraday Components Calculation 
        def compute_intra_components(daily_bars):
            if len(daily_bars) < 2: 
                return pd.Series(dtype=float)

            if freq != "1min":
                # Resample within the session to the target intraday frequency
                resampled = daily_bars.set_index("ts").resample(freq, origin="start").agg({
                    "close": "last", "open": "first"
                }).dropna()
            else:
                resampled = daily_bars.set_index("ts")

            if len(resampled) < 2: return pd.Series(dtype=float)
            if len(resampled) == 0: return pd.Series(dtype=float)

            # Log Returns
            log_close = np.log(resampled["close"])
            ret = log_close.diff().dropna()
            
            # Fallback for extremely short days (erroneous entries, holidays, etc.)
            if len(ret) == 0: 
                rv = 0.0
                bv = 0.0
                jump = 0.0
                semi_up = 0.0
                semi_down = 0.0
                rq = 0.0
            else:
                # 1. Realized Volatility (RV)
                rv = np.sum(np.square(ret))
                
                # 2. Bipower Variation (BV)
                abs_ret = np.abs(ret)
                bv = (np.pi / 2) * np.sum(abs_ret * abs_ret.shift(1).fillna(0))
                
                # 3. Jump Component
                jump = max(rv - bv, 0)
                
                # 4. Semivariances
                semi_up = np.sum(np.square(ret[ret > 0]))
                semi_down = np.sum(np.square(ret[ret < 0]))
                
                # 5. Quarticity
                M = len(ret)
                rq = (M / 3.0) * np.sum(np.power(ret, 4))
            
            return pd.Series({
                "rv_intra": rv, 
                "bv_intra": bv, 
                "jump_intra": jump,
                "semi_up_intra": semi_up, 
                "semi_down_intra": semi_down, 
                "rq_intra": rq,
                "px_open_session": daily_bars["open"].iloc[0],
                "px_close_session": daily_bars["close"].iloc[-1]
            })

        print(f"Calculating Intraday Components ({freq})...")
        daily = df_clean.groupby("day").apply(compute_intra_components)
        
        # Gap Components 
        daily["prev_close"] = daily["px_close_session"].shift(1)
        daily = daily.dropna(subset=["prev_close"])
        
        daily["gap_ret"] = np.log(daily["px_open_session"] / daily["prev_close"])
        daily["gap_sq"] = np.square(daily["gap_ret"])
        
        daily["rv_gap"] = daily["gap_sq"]
        daily["semi_up_gap"] = np.where(daily["gap_ret"] > 0, daily["gap_sq"], 0.0)
        daily["semi_down_gap"] = np.where(daily["gap_ret"] < 0, daily["gap_sq"], 0.0)

        # Total Variance (Target) 
        daily["rv_total"] = daily["rv_intra"] + daily["rv_gap"]
        daily["daily_ret"] = np.log(daily["px_close_session"] / daily["prev_close"])

        # EWMA(0.95) Baseline
        daily["ewma_forecast"] = daily["rv_total"].ewm(alpha=0.05, adjust=False).mean().shift(1)

        # Feature Engineering 
        eps = 1e-9
        
        # 1. Targets
        daily["target_log_rv"] = np.log(daily["rv_total"].shift(-1) + eps)
        daily["target_true_rv"] = daily["rv_total"].shift(-1)

        # 2. Feature Construction
        base_components = [
            "rv_intra", "bv_intra", "rv_gap",
            "semi_up_intra", "semi_down_intra",
            "semi_up_gap", "semi_down_gap",
            "jump_intra", "rq_intra"
        ]
        
        features = []
        for c in base_components:
            log_c = f"log_{c}"
            shift_val = 1.0 if "jump" in c else eps
            # Log transform
            daily[log_c] = np.log(daily[c] + shift_val)
            
            # Lags
            daily[f"{log_c}_d"] = daily[log_c]
            daily[f"{log_c}_w"] = daily[log_c].rolling(5).mean()
            daily[f"{log_c}_m"] = daily[log_c].rolling(22).mean()
            
            features.extend([f"{log_c}_d", f"{log_c}_w", f"{log_c}_m"])

            # Fractal Lags
            if "rv" in c: 
                for lag in [2, 4, 8, 16]:
                    col_name = f"{log_c}_lag{lag}"
                    daily[col_name] = daily[log_c].rolling(lag).mean()
                    features.append(col_name)
        
        # Final Cleanup
        # 1. Replace Infinite values (from log(0)) with NaN
        daily = daily.replace([np.inf, -np.inf], np.nan)
        # 2. Drop any row with NaN (due to lags or previous infs)
        daily = daily.dropna()
        
        print(f"Feature Engineering Done. Total Features: {len(features)}. Valid Days: {len(daily)}")
        return daily, features


## Modeling

We only need to specify complex models; the logic for simpler models (HAR, EWMA, RV) will be in subsequent cells.

### Deep Learning Model

In [3]:
class DeepHAR(nn.Module):
    def __init__(self, input_dim, mean_log_target=-9.0, hidden_dim=64, dropout=0.25):
        super(DeepHAR, self).__init__()
        self.linear_skip = nn.Linear(input_dim, 1)
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.SiLU(), 
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.BatchNorm1d(hidden_dim // 2),
            nn.SiLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )
        # Init bias to mean_log_target to prevent explosion
        nn.init.xavier_uniform_(self.linear_skip.weight, gain=0.1)
        nn.init.constant_(self.linear_skip.bias, mean_log_target)
        for m in self.net.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        har_pred = self.linear_skip(x)
        nn_res = self.net(x)
        return har_pred + nn_res


## Evaluation

In [4]:
class QLIKELoss(nn.Module):
    def __init__(self, eps=1e-6):
        super(QLIKELoss, self).__init__()
        self.eps = eps
        
    def forward(self, pred_log_var, true_var):
        # Pred is log(sigma^2), true is sigma^2
        # QLIKE = log(sigma^2) + true_sigma^2 / sigma^2 = pred_log_var + true_var / exp(pred_log_var)
        
        # Numerical stability: Clamp pred_log_var to avoid exp() -> 0 or exp() -> inf
        pred_log_var = torch.clamp(pred_log_var, min=-20.0, max=20.0)
        
        # Add epsilon to denominator just in case
        loss = pred_log_var + (true_var / (torch.exp(pred_log_var) + self.eps))
        return torch.mean(loss)

In [5]:
def run_ml_tournament(bars_df, input_freq="1min", target_freq="1min", train_years=3, test_months=12):
    # Truncate bars after Oct 2025 to keep a consistent 36-month test window ending then.
    cutoff_ts = pd.Timestamp("2025-10-31 23:59:59", tz="UTC")
    if "ts" in bars_df.columns:
        bars_df = bars_df[bars_df["ts"] <= cutoff_ts].copy()
    elif isinstance(bars_df.index, pd.DatetimeIndex):
        bars_df = bars_df.loc[bars_df.index <= cutoff_ts].copy()

    vol_eng = VolatilityFeatures()
    df, feature_cols = vol_eng.build_daily_table(bars_df, freq=target_freq)
    
    # Fallback
    if df.empty:
        print(f"Skipping input={input_freq}, target={target_freq} - No data generated.")
        return pd.DataFrame()

    # Feature Groups 
    # 1. Log-HAR: Baseline (Standard Daily, Weekly, Monthly RV)
    har_cols = [c for c in feature_cols if ("rv_intra" in c or "rv_gap" in c) and "lag" not in c]
    
    # 2. HAR-RS-CJ-Q: semivariance + jump + quarticity (linear, all features)
    shar_cols = [c for c in feature_cols if "semi_" in c]
    
    # 3. ALL features (for ML models)
    all_feature_cols = list(range(len(feature_cols)))

    print(f"\nTournament Setup \nInput frequency: {input_freq}, Target frequency: {target_freq}\nTotal Data: {len(df)} days")
    
    results = []

    model_names = ["RV", "Log-HAR", "HAR-RS-CJ-Q", "XGBoost", "DeepHAR"]
    train_r2_scores = {m: [] for m in model_names}
    
    scaler_X = StandardScaler()
    
    days_total = len(df)
    test_days = test_months * 21
    train_days = train_years * 252 
    refit_step = 63 # Three months per refit window
    
    # Evaluate on the last `test_months` months when available.
    test_start_idx = max(days_total - test_days, 0)
    current_t = test_start_idx
    
    # Dynamic training window: use up to `train_years` of history, or all available history if shorter.
    while current_t < days_total:
        train_end = current_t
        train_start = max(0, train_end - train_days)
        test_end = min(current_t + refit_step, days_total)
        

        print(f"Refitting: {df.index[train_end].date()} -> {df.index[test_end-1].date()}")

        # Slicing & Safe Handling 
        X_tr_raw = df[feature_cols].iloc[train_start:train_end]
        y_tr_log = df["target_log_rv"].iloc[train_start:train_end]
        y_tr_true = df["target_true_rv"].iloc[train_start:train_end]
        
        if X_tr_raw.isna().any().any() or y_tr_log.isna().any():
            X_tr_raw = X_tr_raw.ffill().bfill().fillna(0)
            y_tr_log = y_tr_log.ffill().bfill().fillna(-9.0)

        mean_log_target = y_tr_log.mean()
        if pd.isna(mean_log_target) or np.isinf(mean_log_target):
            mean_log_target = -9.0 
        
        X_te_raw = df[feature_cols].iloc[train_end:test_end]
        y_te_true = df["target_true_rv"].iloc[train_end:test_end]
        dates_te = df.index[train_end:test_end]
        
        # Baselines
        ewma_pred = df["ewma_forecast"].iloc[train_end:test_end].values
        try:
            ret_tr = df["daily_ret"].iloc[train_start:train_end].values
            ret_te = df["daily_ret"].iloc[train_end:test_end].values
            if len(ret_tr) < 2 or np.var(ret_tr) < 1e-12:
                raise ValueError("Insufficient return variation for GARCH fit")
            scale = 100.0
            r_tr = ret_tr * scale
            r_all = np.concatenate([ret_tr, ret_te]) * scale
            am_tr = arch_model(r_tr, mean="Zero", vol="Garch", p=1, q=1, dist="normal")
            res_tr = am_tr.fit(disp="off")
            params = res_tr.params
            am_all = arch_model(r_all, mean="Zero", vol="Garch", p=1, q=1, dist="normal")
            res_all = am_all.fix(params)
            vars_all = np.square(res_all.conditional_volatility)
            garch_pred = vars_all[len(r_tr):] / (scale**2)
        except Exception:
            garch_pred = ewma_pred

        # Scaling
        try:
            X_tr_scaled = scaler_X.fit_transform(X_tr_raw)
            X_te_scaled = scaler_X.transform(X_te_raw)
            # NaN safety after scaling
            X_tr_scaled = np.nan_to_num(X_tr_scaled)
            X_te_scaled = np.nan_to_num(X_te_scaled)
        except ValueError:
            current_t += refit_step
            continue
        
        batch_preds = {}

        # 1. Ridge Models (Specific Features) 
        ridge_configs = {
            "RV": [feature_cols.index("log_rv_intra_d")] if "log_rv_intra_d" in feature_cols else [],
            "Log-HAR": [feature_cols.index(c) for c in har_cols],
            "HAR-RS-CJ-Q": all_feature_cols
        }

        
        for m_name, idxs in ridge_configs.items():
            if len(idxs) == 0: continue
            reg = Ridge(alpha=1.0).fit(X_tr_scaled[:, idxs], y_tr_log)
            batch_preds[m_name] = np.exp(reg.predict(X_te_scaled[:, idxs]))
            train_r2_scores[m_name].append(r2_score(y_tr_true, np.exp(reg.predict(X_tr_scaled[:, idxs]))))
            
        # ML Models: Use all features
        X_tr_ks = X_tr_scaled[:, all_feature_cols]
        X_te_ks = X_te_scaled[:, all_feature_cols]

        # 2. XGBoost (all features)
        dtrain = xgb.DMatrix(X_tr_ks, label=y_tr_log)
        dtest = xgb.DMatrix(X_te_ks)
        params_xgb = {'objective': 'reg:squarederror', 'max_depth': 3, 'eta': 0.05, 'subsample': 0.7}
        bst = xgb.train(params_xgb, dtrain, 200)
        batch_preds["XGBoost"] = np.exp(bst.predict(dtest))
        train_r2_scores["XGBoost"].append(r2_score(y_tr_true, np.exp(bst.predict(dtrain))))
        
        #  3. DeepHAR (all features) 
        X_tr_t = torch.FloatTensor(X_tr_ks).to(device)
        y_tr_t = torch.FloatTensor(y_tr_true.values).unsqueeze(1).to(device)
        X_te_t = torch.FloatTensor(X_te_ks).to(device)
        
        dh_model = DeepHAR(X_tr_ks.shape[1], mean_log_target=mean_log_target).to(device)
        crit = QLIKELoss()
        opt = optim.AdamW(dh_model.parameters(), lr=0.001, weight_decay=1e-4)
        
        ds = TensorDataset(X_tr_t, y_tr_t)
        if len(ds) >= 2:
            bs = min(64, len(ds))
            dl = DataLoader(ds, batch_size=bs, shuffle=True, drop_last=(len(ds) > bs))
            
            dh_model.train()
            for epoch in range(30): 
                for xb, yb in dl:
                    opt.zero_grad()
                    pred_log = dh_model(xb)
                    loss = crit(pred_log, yb)
                    if torch.isfinite(loss):
                        loss.backward()
                        torch.nn.utils.clip_grad_norm_(dh_model.parameters(), 1.0)
                        opt.step()
                    
            dh_model.eval()
            with torch.no_grad():
                dh_pred = dh_model(X_te_t).cpu().numpy().flatten()
                dh_pred = np.nan_to_num(dh_pred, nan=mean_log_target)
                batch_preds["DeepHAR"] = np.exp(dh_pred)
                
                tr_p = dh_model(X_tr_t).cpu().numpy().flatten()
                tr_p = np.nan_to_num(tr_p, nan=mean_log_target)
                valid_mask = np.isfinite(y_tr_true.values) & np.isfinite(tr_p)
                if valid_mask.sum() > 2:
                    train_r2_scores["DeepHAR"].append(r2_score(y_tr_true.values[valid_mask], np.exp(tr_p[valid_mask])))
                else:
                    train_r2_scores["DeepHAR"].append(0.0)
        else:
            # Fallback: if the NN cannot be trained (tiny window), reuse Log-HAR or EWMA.
            if "Log-HAR" in batch_preds:
                batch_preds["DeepHAR"] = batch_preds["Log-HAR"]
            else:
                batch_preds["DeepHAR"] = ewma_pred
            train_r2_scores["DeepHAR"].append(0.0)

        for i, dt in enumerate(dates_te):
            row = {"date": dt, "True": y_te_true.iloc[i], "EWMA": ewma_pred[i], "GARCH": garch_pred[i]}
            for m in batch_preds:
                if i < len(batch_preds[m]): row[m] = batch_preds[m][i]
            results.append(row)
        current_t += refit_step

    # Evaluation
    if not results:
        print("No results generated.")
        return pd.DataFrame()

    res_df = pd.DataFrame(results).set_index("date")
    metrics = []
    all_models = ["EWMA", "GARCH", "RV", "Log-HAR", "HAR-RS-CJ-Q", "XGBoost", "DeepHAR"]
    
    print("\n Final Leaderboard ")
    for m in all_models:
        if m not in res_df.columns: continue
        y_true, y_pred = res_df["True"], res_df[m]
        valid = np.isfinite(y_pred) & (y_pred > 0)
        y_t, y_p = y_true[valid], y_pred[valid]
        if len(y_p) == 0: continue

        # Log Transformation for Mincer-Zarnowitz R^2 
        log_y_t = np.log(y_t)
        log_y_p = np.log(y_p)
        
        metrics.append({
            "Model": m,
            "QLIKE": np.mean(np.log(y_p) + y_t/y_p),
            "Log Test R2": r2_score(log_y_t, log_y_p)
        })
        
    lb = pd.DataFrame(metrics).sort_values("QLIKE")
    print(lb.to_string(index=False, float_format=lambda x: "{:.6f}".format(x) if abs(x)<1 else "{:.4f}".format(x)))
    
    return lb


In [6]:
# Load 1-minute OHLCV data for the three instruments
instruments = {
    "ES": "data/es_ohlcv_1m.parquet",
    "6E": "data/6e_ohlcv_1m.parquet",
    "GC": "data/gc_ohlcv_1m.parquet",
}

bars_1m_by_inst = {k: pd.read_parquet(v) for k, v in instruments.items()}

def resample_bars(bars: pd.DataFrame, freq: str) -> pd.DataFrame:
    if freq == "1min":
        return bars
    if "ts" in bars.columns:
        b = bars.copy()
        b["ts"] = pd.to_datetime(b["ts"], utc=True)
    elif isinstance(bars.index, pd.DatetimeIndex):
        b = bars.copy().reset_index().rename(columns={bars.index.name or "index": "ts"})
        b["ts"] = pd.to_datetime(b["ts"], utc=True)
    else:
        raise ValueError("Could not infer timestamps for resampling.")

    return (
        b.set_index("ts")
        .resample(freq, origin="start")
        .agg({"open": "first", "close": "last"})
        .dropna()
        .reset_index()
    )

bars_5m_by_inst = {k: resample_bars(bars, "5min") for k, bars in bars_1m_by_inst.items()}

In [7]:
# Run 1m and 5m sampling intervals per instrument
leaderboards = {}
for inst in instruments.keys():
    print(f"\n=== {inst} ===")
    bars_1m = bars_1m_by_inst[inst]
    bars_5m = bars_5m_by_inst[inst]

    print("\n=== 1m sampling (1m input -> 1m target) ===")
    lb_1m = run_ml_tournament(bars_1m, input_freq="1min", target_freq="1min", train_years=12, test_months=36)

    print("\n=== 5m sampling (5m input -> 5m target) ===")
    lb_5m = run_ml_tournament(bars_5m, input_freq="5min", target_freq="5min", train_years=12, test_months=36)

    leaderboards[inst] = {"1m": lb_1m, "5m": lb_5m}



=== ES ===

=== 1m sampling (1m input -> 1m target) ===
Maintenance Filter: Dropped 24659 bars found between 17:00-17:59 ET.
Calculating Intraday Components (1min)...
Feature Engineering Done. Total Features: 35. Valid Days: 3943

Tournament Setup 
Input frequency: 1min, Target frequency: 1min
Total Data: 3943 days
Refitting: 2022-11-28 -> 2023-02-24
Refitting: 2023-02-27 -> 2023-05-24
Refitting: 2023-05-25 -> 2023-08-21
Refitting: 2023-08-22 -> 2023-11-16
Refitting: 2023-11-17 -> 2024-02-15
Refitting: 2024-02-16 -> 2024-05-15
Refitting: 2024-05-16 -> 2024-08-12
Refitting: 2024-08-13 -> 2024-11-07
Refitting: 2024-11-08 -> 2025-02-06
Refitting: 2025-02-07 -> 2025-05-07
Refitting: 2025-05-08 -> 2025-08-04
Refitting: 2025-08-05 -> 2025-10-30

 Final Leaderboard 
      Model   QLIKE  Log Test R2
    DeepHAR -8.5915     0.475859
    XGBoost -8.5802     0.514112
HAR-RS-CJ-Q -8.5758     0.521306
    Log-HAR -8.5609     0.497386
         RV -8.5374     0.448161
      GARCH -8.5049     0.21184